In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Corridor Transit Profiles on the V2 Routes — Calibrated Survey vs Ticketing-Based

Step 18 compared the calibrated-survey transit profile with the ticketing-based one on the earlier 18-area line. This notebook repeats the comparison on the **V2 aggregation** (25 areas, routes T1 / T2 / T3 and the tree network of step 24), from the TAZ-level products so that no earlier area aggregation is involved:

| Profile | Bus layer | Rail layer | Frame |
|---|---|---|---|
| **Calibrated survey** — `Output/ths2017/three_mode_2022/bus_2022_taz.csv`, `rail_2022_taz.csv` | survey Public Bus + Matronit, destination pattern blended with RavKav's own alightings, RavKav volumes where ticketing coverage is credible, grown to 2022 where not | survey rail, door-to-door | residents, doorstep origins; 2022 |
| **Ticketing-based** — `Output/bus/bus_od_taz_avg.csv`, `Output/train/train_od_taz_6_9.csv` × 0.793 | RavKav journeys on RavKav's own inferred alightings, May 2022 (revised 23 September 2026: the OnBoard-patterned matrix of step 9 is retired as the reference, METHODOLOGY §6ag) | 2019 smartcard station-to-station, levelled to 2022 | all riders, boarding-stop origins; 2022 |

Transit is bus + rail in both sets; the survey's taxi-type layer is shown alongside and never added. The calibration's intermediate steps (raw 2018 survey bus, all-RavKav variant) are drawn as before. Every value is a three-hour total of potential movements between line areas, not a load.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/corridor_v2'; os.makedirs('Output/figures', exist_ok=True)
xl = pd.ExcelFile('Input/Corridor_TAZ_Agg_V2.xlsx'); areas = xl.parse('AreaCodes').set_index('AggCode'); key = xl.parse('TazAgg')
area_of = key.set_index('TAZ')['AggCode']; AREAS = list(areas.index); names = areas['AggAreaName']
ROUTES = {r: list(areas[areas[f'Order_{r}'] > 0].sort_values(f'Order_{r}').index) for r in ['T1', 'T2', 'T3']}
TRUNK = [a for a in ROUTES['T1'] if all(a in ROUTES[r] for r in ROUTES)]; BRANCH = {r: [a for a in seq if a not in TRUNK] for r, seq in ROUTES.items()}
def load(path):
    m = pd.read_csv(path, index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int); return m
def to_area(m):
    t = m.loc[m.index.isin(area_of.index), m.columns.isin(area_of.index)]
    t = t.groupby(t.index.map(area_of)).sum().T.groupby(lambda c: area_of[c]).sum().T
    return t.reindex(index=AREAS, columns=AREAS).fillna(0)
IN = 'Output/ths2017/three_mode_2022'
H_bus, H_taxi, H_rail = to_area(load(f'{IN}/bus_2022_taz.csv')), to_area(load(f'{IN}/taxi_2022_taz.csv')), to_area(load(f'{IN}/rail_2022_taz.csv'))
H_total = to_area(load(f'{IN}/car_2022_taz.csv')) + H_bus + H_taxi + H_rail
T_bus = to_area(load('Output/bus/bus_od_taz_avg.csv'))   # ticketing reference: RavKav journeys on RavKav's own alightings (revised 23 Sep 2026, §6ag)
T_train = to_area(load('Output/train/train_od_taz_6_9.csv')) * (54.7 / 69.0)
S_bus18 = to_area(load('Output/ths2017/two_mode/bus_survey_taz.csv')); R_bus = to_area(load('Output/ths2017/two_mode/bus_calibrated_all_ravkav_taz.csv'))
COMP = {'survey: bus (calibrated)': H_bus, 'survey: rail': H_rail, 'survey: taxi-type': H_taxi, 'ticketing: bus (RavKav, own alightings)': T_bus, 'ticketing: train (station)': T_train,
        'step: survey bus 2018 (raw)': S_bus18, 'step: bus, all-RavKav volumes': R_bus, 'survey: total (car + bus + taxi + rail)': H_total}
print("trips with both ends in the 25 V2 areas — " + ', '.join(f'{k} {v.values.sum():,.0f}' for k, v in COMP.items()))

trips with both ends in the 25 V2 areas — survey: bus (calibrated) 22,735, survey: rail 51, survey: taxi-type 2,170, ticketing: bus (RavKav, own alightings) 18,540, ticketing: train (station) 301, step: survey bus 2018 (raw) 20,620, step: bus, all-RavKav volumes 17,276, survey: total (car + bus + taxi + rail) 181,900


## 1. Link profiles per route and on the tree network

In [3]:
def link_flows(m, seq):
    sub = m.reindex(index=seq, columns=seq).fillna(0).values; n = len(seq); up, down = np.zeros(n - 1), np.zeros(n - 1)
    for i in range(n):
        for j in range(n):
            if i < j: up[i:j] += sub[i, j]
            elif i > j: down[j:i] += sub[i, j]
    return up, down
# tree paths for the network view (as in step 24)
LINKS = [(TRUNK[i], TRUNK[i + 1]) for i in range(len(TRUNK) - 1)] + [(([TRUNK[-1]] + br)[i], ([TRUNK[-1]] + br)[i + 1]) for br in BRANCH.values() for i in range(len(br))]
lpos = {l: i for i, l in enumerate(LINKS)}
def to_root(a):
    if a in TRUNK: return list(reversed(TRUNK[:TRUNK.index(a) + 1]))
    r = next(r for r in ROUTES if a in BRANCH[r]); i = BRANCH[r].index(a)
    return list(reversed(BRANCH[r][:i + 1])) + list(reversed(TRUNK))
def path_links(o, d_):
    if o == d_: return []
    po, pd_ = to_root(o), to_root(d_); common = next(a for a in po if a in set(pd_))
    down_part = po[:po.index(common) + 1]; up_part = list(reversed(pd_[:pd_.index(common) + 1]))
    return [(lpos[(down_part[i + 1], down_part[i])], 'down') for i in range(len(down_part) - 1)] + [(lpos[(up_part[i], up_part[i + 1])], 'up') for i in range(len(up_part) - 1)]
PATHS = {(o, d_): path_links(o, d_) for o in AREAS for d_ in AREAS}
def network_flows(m):
    F = {'up': np.zeros(len(LINKS)), 'down': np.zeros(len(LINKS))}; sub = m.reindex(index=AREAS, columns=AREAS).values
    for i, o in enumerate(AREAS):
        for j, d_ in enumerate(AREAS):
            for li, dd in PATHS[(o, d_)]: F[dd][li] += sub[i, j]
    return F['up'], F['down']
rows = []
for scope in ['T1', 'T2', 'T3', 'network']:
    if scope == 'network': links = LINKS
    else: seq = ROUTES[scope]; links = list(zip(seq[:-1], seq[1:]))
    LF = {k: (network_flows(v) if scope == 'network' else link_flows(v, ROUTES[scope])) for k, v in COMP.items()}
    for li, (a, b) in enumerate(links):
        for di, dname in [(0, 'up'), (1, 'down')]:
            r = {'scope': scope, 'link': li + 1, 'from_area': names[a], 'to_area': names[b], 'segment': 'trunk' if b in TRUNK else 'branch', 'direction': dname}
            for k in COMP: r[k] = LF[k][di][li]
            r['survey transit'] = r['survey: bus (calibrated)'] + r['survey: rail']; r['ticketing transit'] = r['ticketing: bus (RavKav, own alightings)'] + r['ticketing: train (station)']
            r['ratio survey / ticketing'] = r['survey transit'] / r['ticketing transit'] if r['ticketing transit'] > 0 else np.nan
            r['transit share (survey set)'] = r['survey transit'] / r['survey: total (car + bus + taxi + rail)'] if r['survey: total (car + bus + taxi + rail)'] > 0 else np.nan
            rows.append(r)
cmp = pd.DataFrame(rows); cmp.to_csv(f'{OUT}/corridor_v2_survey_vs_ticketing.csv', index=False, float_format='%.2f')
seg = []
for scope in ['T1', 'T2', 'T3', 'network']:
    for dname in ['up', 'down']:
        s = cmp[(cmp.scope == scope) & (cmp.direction == dname)]; hs = s[s.link <= 9]; ks = s[s.link >= 10]     # Haifa segment: Tirat Carmel … Hamifrats (links 1–9); east / north of Hamifrats: links 10+
        seg.append({'scope': scope, 'direction': dname, 'survey peak': s['survey transit'].max(), 'survey peak link': f"{s.loc[s['survey transit'].idxmax(), 'from_area']} – {s.loc[s['survey transit'].idxmax(), 'to_area']}",
                    'ticketing peak': s['ticketing transit'].max(), 'ticketing peak link': f"{s.loc[s['ticketing transit'].idxmax(), 'from_area']} – {s.loc[s['ticketing transit'].idxmax(), 'to_area']}",
                    'Haifa segment survey / ticketing': hs['survey transit'].sum() / hs['ticketing transit'].sum(), 'beyond Hamifrats survey / ticketing': ks['survey transit'].sum() / ks['ticketing transit'].sum() if ks['ticketing transit'].sum() > 0 else np.nan})
seg = pd.DataFrame(seg); seg.to_csv(f'{OUT}/corridor_v2_survey_vs_ticketing_summary.csv', index=False, float_format='%.3f'); print(seg.round(2).to_string(index=False))
cmp[cmp.scope == 'network'][['link', 'from_area', 'to_area', 'direction', 'survey transit', 'ticketing transit', 'ratio survey / ticketing', 'survey: taxi-type', 'transit share (survey set)']].round(2)

  scope direction  survey peak                       survey peak link  ticketing peak               ticketing peak link  Haifa segment survey / ticketing  beyond Hamifrats survey / ticketing
     T1        up      1688.92      EinHayam – BatGalim-KiryatEliezer         1429.74 EinHayam – BatGalim-KiryatEliezer                              1.02                                 0.88
     T1      down      1352.39 HofCarmel-NeveDavid – Hecht-Shprintzak         1210.43         LowerCity – Namal-Giborim                              1.04                                 1.55
     T2        up      1769.30      EinHayam – BatGalim-KiryatEliezer         1506.99 EinHayam – BatGalim-KiryatEliezer                              1.01                                 0.99
     T2      down      2297.75           TsometKiryatAta – KiryatHaim         1854.68         LowerCity – Namal-Giborim                              1.12                                 1.24
     T3        up      1712.49      EinHayam 

,link,from_area,to_area,direction,survey transit,ticketing transit,ratio survey / ticketing,survey: taxi-type,transit share (survey set)
92,1,TiratCarmel,Matam-NeotPeres,up,930.75,952.50,0.98,0.00,0.19
93,1,TiratCarmel,Matam-NeotPeres,down,521.21,618.50,0.84,0.00,0.30
94,2,Matam-NeotPeres,HofCarmel-NeveDavid,up,937.42,884.56,1.06,0.00,0.26
95,2,Matam-NeotPeres,HofCarmel-NeveDavid,down,1936.65,1694.84,1.14,96.02,0.32
96,3,HofCarmel-NeveDavid,Hecht-Shprintzak,up,1109.36,1098.06,1.01,0.00,0.30
97,3,HofCarmel-NeveDavid,Hecht-Shprintzak,down,1995.65,1811.59,1.10,61.36,0.33
98,4,Hecht-Shprintzak,EinHayam,up,1648.38,1507.06,1.09,0.00,0.33
99,4,Hecht-Shprintzak,EinHayam,down,1853.65,1647.84,1.12,39.52,0.32
100,5,EinHayam,BatGalim-KiryatEliezer,up,1865.11,1549.31,1.20,0.00,0.31
101,5,EinHayam,BatGalim-KiryatEliezer,down,1747.14,1654.09,1.06,0.00,0.32


In [4]:
fig, axes = plt.subplots(4, 2, figsize=(17, 20), facecolor='white')
for i, scope in enumerate(['T1', 'T2', 'T3', 'network']):
    if scope == 'network': labels = [f'{names[a]} – {names[b]}' for a, b in LINKS]; n = len(LINKS)
    else: seq = ROUTES[scope]; labels = [f'{a} · {names[a]}' for a in seq]; n = len(seq) - 1
    for j, dname in enumerate(['up', 'down']):
        ax = axes[i, j]; s = cmp[(cmp.scope == scope) & (cmp.direction == dname)].sort_values('link'); x = np.arange(n + 1)
        for k, color, ls, lw in [('step: survey bus 2018 (raw)', MUTED, ':', 1.5), ('step: bus, all-RavKav volumes', PURPLE, '--', 1.3), ('survey: bus (calibrated)', BLUE, '-', 2.2), ('ticketing: bus (RavKav, own alightings)', ORANGE, '-', 2.2)]:
            ax.stairs(s[k].values, x, color=color, linestyle=ls, linewidth=lw, label=k)
        ax.stairs(s['survey transit'].values, x, color=BLUE, linewidth=1, linestyle='-.', alpha=0.7, label='survey transit (bus + rail)')
        ax.stairs(s['ticketing transit'].values, x, color=ORANGE, linewidth=1, linestyle='-.', alpha=0.7, label='ticketing transit (bus + train)')
        ax.stairs(s['survey: taxi-type'].values, x, color=AQUA, linewidth=1.2, linestyle=':', label='survey taxi-type (separate)')
        if scope != 'network': ax.axvspan(len(TRUNK) - 1, n, color=MUTED, alpha=0.07, lw=0)
        ax.set_xlim(0, n); ax.grid(True, color=GRID, linewidth=0.6, axis='y'); ax.set_axisbelow(True)
        for sp in ax.spines.values(): sp.set_color(AXIS)
        ax.tick_params(colors=INK2); ax.set_ylabel('trips crossing the link, 06:00–09:00', color=INK2, fontsize=9)
        if scope == 'network': ax.set_xticks(x[:-1] + 0.5); ax.set_xticklabels(labels, rotation=70, ha='right', fontsize=6.5, color=INK2)
        else: ax.set_xticks(x); ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=7.5, color=INK2)
        ax.set_title(f'{scope if scope != "network" else "tree network (24 links)"} — {dname} ({"away from" if dname == "up" else "towards"} Tirat Carmel)', color=INK, fontsize=11)
        if i == 0: ax.legend(frameon=False, fontsize=7.5, loc='upper right')
fig.suptitle('Corridor bus / transit potential movements on the V2 routes — calibrated survey vs ticketing, with the calibration steps (3-hour totals, not loads)', color=INK, fontsize=13, y=1.0)
plt.tight_layout(); fig.savefig('Output/figures/corridor_v2_survey_vs_ticketing.png', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()

## 2. Which area pairs make the difference (tree network, link-trips)

In [5]:
Hm, Tm = (H_bus + H_rail), (T_bus + T_train); rows = []
for o in AREAS:
    for d_ in AREAS:
        if o == d_: continue
        pl = PATHS[(o, d_)]; nl = len(pl); ndown = sum(1 for _, dd in pl if dd == 'down')
        rows.append({'origin': names[o], 'destination': names[d_], 'origin segment': 'trunk' if o in TRUNK else next(r for r in ROUTES if o in BRANCH[r]), 'destination segment': 'trunk' if d_ in TRUNK else next(r for r in ROUTES if d_ in BRANCH[r]),
                     'links crossed': nl, 'links down': ndown, 'survey trips': Hm.loc[o, d_], 'ticketing trips': Tm.loc[o, d_], 'diff trips': Hm.loc[o, d_] - Tm.loc[o, d_], 'diff link-trips': (Hm.loc[o, d_] - Tm.loc[o, d_]) * nl})
pairs = pd.DataFrame(rows); pairs.to_csv(f'{OUT}/corridor_v2_survey_vs_ticketing_pairs.csv', index=False, float_format='%.1f')
print("largest contributions to the difference (survey − ticketing), link-trips on the tree network:")
print(pd.concat([pairs.sort_values('diff link-trips').head(10), pairs.sort_values('diff link-trips').tail(8)]).round(0).to_string(index=False))
by_o = pairs.groupby('origin')[['survey trips', 'ticketing trips', 'diff link-trips']].sum().sort_values('diff link-trips'); by_d = pairs.groupby('destination')[['survey trips', 'ticketing trips', 'diff link-trips']].sum().sort_values('diff link-trips')
print("\nby origin area:"); print(by_o.round(0).to_string()); print("\nby destination area:"); print(by_d.round(0).to_string())

largest contributions to the difference (survey − ticketing), link-trips on the tree network:
                origin            destination origin segment destination segment  links crossed  links down  survey trips  ticketing trips  diff trips  diff link-trips
         KiryatYam B+C BatGalim-KiryatEliezer             T3               trunk              8           8          72.0            269.0      -197.0          -1578.0
         KiryatYam B+C        Matam-NeotPeres             T3               trunk             12          12          72.0            135.0       -64.0           -763.0
         KiryatYam B+C              LowerCity             T3               trunk              6           6          69.0            166.0       -97.0           -581.0
       KiryatHaim West BatGalim-KiryatEliezer             T3               trunk              7           7          57.0            138.0       -81.0           -570.0
           TiratCarmel              Hamifrats          trunk      

## Findings

- **Totals in the 25 V2 areas.** Calibrated survey transit 18,839 (bus 18,788 + rail 51) against ticketing 16,363 (bus 16,062 + train 301); the raw 2018 survey bus 14,831 and the all-RavKav variant 14,276. The survey set carries 15 % more transit trips than the ticketing set overall, yet sits below it on almost every link — the survey's transit is more local (short pairs that cross few links), the ticketing's more corridor-long.
- **Up direction (away from Tirat Carmel) agrees as in step 18.** On the Haifa segment the survey profile is 0.78–0.83 × the ticketing one per route (0.82 on the network), and both peak on the same link, Ein Hayam – Bat Galim-Kiryat Eliezer (survey 1,379–1,437, ticketing 1,556–1,646; network 1,518 vs 1,730). Beyond Hamifrats the routes differ: T1 (Nazareth branch) 1.08, T2 / T3 (Krayot, Kiryat Yam) 0.57–0.58.
- **Down direction (into Haifa) is where the sets still disagree, and the gap is route-specific.** Haifa segment survey / ticketing: T2 0.92, T3 0.73, T1 0.66, network 0.63; beyond Hamifrats T2 / T3 0.79, T1 0.43. The ticketing peak sits on Lower City – Namal-Giborim per route (2,073–2,903) and on Bazan-Hutsot – Tsomet Kiryat Ata on the network (4,476 vs survey 2,898); the survey peak is further west, Matam-Neot Peres – Hof Carmel-Neve David (1,410–1,629).
- **The drivers are the same pairs as before, now named on the V2 areas.** Nazareth as origin: 1,070 survey vs 2,402 ticketed corridor-bound transit trips (−13,020 link-trips), above all Nazareth → Bat Galim-Kiryat Eliezer (87 vs 521), → Hecht-Shprintzak (22 vs 271), → Lower City (132 vs 431), → Hamifrats (50 vs 283). Then Kiryat Yam B+C (568 vs 1,116), Tsur Shalom (290 vs 642) and the Hamifrats hub (431 vs 1,049) as origins. By destination the ticketing set lands far more at Bat Galim-Kiryat Eliezer (2,064 vs 1,161), Lower City (1,764 vs 894) and Hecht-Shprintzak (1,035 vs 279) — trunk-route alighting stops — while the survey places more at Kiryat Ata North, Kiryat Yam B+C and Bazan-Hutsot and sends more from Kiryat Bialik Center, Ein Hayam and Hecht-Shprintzak. The diagnosis of step 18 holds on the new geography: the Nazareth-to-Haifa allocation and the destination frame (employment-share TAZs vs alighting stops), plus hub attribution at Hamifrats, not ticketing coverage.
- **What this means for the V2 profiles.** The T2 route is the one where the two frames nearly agree in both directions; the T1 route is the one where the ticketing set puts about 2.3 × the survey's transit on the branch towards Haifa. Any capture-model market on the Nazareth branch should carry both frames as a range until the Nazareth allocation is settled (tasks A1, B1c, B2).
- **Outputs.** `Output/corridor_v2/corridor_v2_survey_vs_ticketing.csv` (per scope, link and direction: every component, the two transit profiles, their ratio, the survey-set transit share), `corridor_v2_survey_vs_ticketing_summary.csv`, `corridor_v2_survey_vs_ticketing_pairs.csv`; figure `corridor_v2_survey_vs_ticketing.png`.